## 本部分内容为grid2op环境的构建与注册，如无特殊需要请勿更改

测试使用l2rpn_wcci_2022环境，该环境中包含所有grid2op当前版本所支持的组件类型

In [1]:
import sys

# 导入grid2op必要组件
import grid2op
from grid2op.PlotGrid import PlotMatplot
from grid2op.Action import PlayableAction
from grid2op.Parameters import Parameters

# 导入自定义组件，包含日志以及注册环境所需全局变量
from registry_object import EnvironmentManager
from src.utils import get_logger

# 以下为在notebook中测试使用
from typing import Dict, List, Any

[2025-02-25 15:03:11,955] [src.utils] [__init__.py(39)] [INFO] 这是带时间戳的日志文件测试信息。


In [2]:
# 设置环境名称
env_name = "l2rpn_wcci_2022"

# 设置环境参数
p = Parameters()
p.MAX_SUB_CHANGED = 3 # 设置MAX_SUB_CHANGED参数的值为3
p.MAX_LINE_STATUS_CHANGED = 3 # 设置MAX_LINE_STATUS_CHANGED参数的值为3

# 初始化日志
logger = get_logger(__name__)

# 创建环境管理器实例用于注册并传递geid2op环境与观测实例
env_manager = EnvironmentManager()

try:
    from lightsim2grid import LightSimBackend
    bk_cls = LightSimBackend
    logger.info("成功导入 LightSimBackend 用于更快的后端计算。")
except ImportError as exc:
    logger.warning(f"导入 LightSimBackend 时发生错误：{exc}，将使用 PandaPowerBackend 作为后备方案。")
    from grid2op.Backend import PandaPowerBackend
    bk_cls = PandaPowerBackend

# 生成唯一环境标识符
env_id = "test1"

# 创建环境
env = grid2op.make(env_name, action_class=PlayableAction, param=p, backend=bk_cls())
logger.info(f"环境 '{env_name}' 创建成功，使用后端: {bk_cls.__name__}。")

# 使用上下文管理器注册环境并处理相关操作
try:
    # 使用上下文管理器注册环境
    with env_manager.register_env(env_id, env):
        try:
            logger.info(f"环境 '{env_name}' 注册成功。")

            # 重置环境并获取初始观察值
            obs = env.reset()

            # 注册初始观察值
            env_manager.record_observation(env_id, obs)
            logger.info(f"环境 '{env_id}' 注册成功并获取初始观察值。")
        except Exception as e:
            logger.error(f"环境 '{env_name}' 注册失败，错误信息：{e}")
            sys.exit(1)

except Exception as e:
    logger.error(f"操作失败，错误信息：{e}")
    sys.exit(1)


[2025-02-25 15:03:11,971] [__main__] [2583467218.py(18)] [INFO] 成功导入 LightSimBackend 用于更快的后端计算。
[2025-02-25 15:03:12,226] [pandapower.convert_format] [convert_format.py(102)] [INFO] These dtypes could not be corrected: {'trafo': ['tap_min', 'tap_max']}
[2025-02-25 15:03:13,371] [__main__] [2583467218.py(29)] [INFO] 环境 'l2rpn_wcci_2022' 创建成功，使用后端: LightSimBackend。
[2025-02-25 15:03:13,373] [__main__] [2583467218.py(36)] [INFO] 环境 'l2rpn_wcci_2022' 注册成功。
[2025-02-25 15:03:13,717] [__main__] [2583467218.py(43)] [INFO] 环境 'test1' 注册成功并获取初始观察值。


## 在此行以后进行必要的功能测试

In [3]:
print(env_manager.get_latest_obs(env_id))

In [4]:
'''
测试impl函数
'''

def line_set_status_impl(env_id: str, line_id: List[int], line_status: List[int]) -> Dict[str, Any]:
    """
    在grid2op环境中设置输电线路的状态。

    参数:
    env_id (str): 环境实例的ID。
    line_id (List[int]): 输电线路ID列表。
    line_status (List[int]): 输电线路状态设置列表，每个整数代表对相应输电线要执行的操作，具体含义如下：
        - 0: 该动作不对这个线路产生作用。
        - 1: 强制连接该线路。
        - -1: 强制断开该线路。

    返回:
    Dict[str, Any]: 包含状态和消息的字典。状态可以是 "success" 或 "failure"，消息描述了操作的结果。

    异常:
    ValueError: 如果环境ID无效或动作是模糊的。
    """
    try:
        logger.info(f"开始执行 line set status 操作，env_id: {env_id}, line_id: {line_id}, line_status: {line_status}")
        
        # 获取环境实例
        env = env_manager._envs.get(env_id)  # 通过 env_manager 获取环境实例
        if env is None:
            raise ValueError(f"无法找到 env_id 为 {env_id} 的环境实例。")

        # 创建动作
        act = env.action_space()
        act.line_set_status = list(zip(line_id, line_status))

        # 检查动作是否模糊
        if act.is_ambiguous()[0]:
            info = act.ambiguous_info()[1]
            raise ValueError(f"动作是模糊的，无法执行。{info}")
        else:
            # 执行动作并获取新的观察
            obs, _, _, _ = env.step(act)
            
            # 使用 EnvironmentManager 注册新观察
            env_manager.record_observation(env_id, obs)  # 新的观察通过 EnvironmentManager 记录
            
            logger.info(f"成功进行 line set status 操作。")
            return {
                "status": "success",
                "message": f"成功进行 line set status 操作。"
            }
    except Exception as e:
        logger.error(f"line_set_status_impl 执行失败: {str(e)}")
        return {
            "status": "failure",
            "message": f"line_set_status_impl 执行失败: {str(e)}"
        }



In [5]:
'''
在此block中进行初始状态的观测
'''

# 以下根据需求修改
obs_before = env_manager.get_latest_obs("test1")
obs_before.line_status[:3]

array([ True,  True,  True])

In [6]:
line_set_status_impl("test1", [0, 1, 2],[0, -1, -1])

[2025-02-25 15:03:13,772] [__main__] [343736242.py(24)] [INFO] 开始执行 line set status 操作，env_id: test1, line_id: [0, 1, 2], line_status: [0, -1, -1]
[2025-02-25 15:03:13,786] [__main__] [343736242.py(46)] [INFO] 成功进行 line set status 操作。


{'status': 'success', 'message': '成功进行 line set status 操作。'}

In [7]:
'''
在此block中验证函数是否正确对组件进行更改
'''

# 以下根据需求修改
obs_after = env_manager.get_latest_obs("test1")
obs_after.line_status[:3]

array([ True, False, False])

In [8]:
'''
绘制前后状态的观测图
'''

# plot_helper = PlotMatplot(env.observation_space)
# fig_before = plot_helper.plot_obs(obs_before)
# fig_after = plot_helper.plot_obs(obs_after)
# fig_before.show()
# fig_after.show()

'\n绘制前后状态的观测图\n'